# Step-1 : Install  the required libraries/packages

In [ ]:
#pip install pandas
#pip install openpyxl
#pip install duckdb


# Step-2 : Import the required libraries/packages

In [42]:
import pandas as pd
import sqlite3
import duckdb
import os 

# Step 3: Load/Import data & Build connection

In [43]:
# -------- File Paths --------
links_file = "Input_files/links.csv.xlsx"
product_file = "Input_files/product_sample_500.csv.xlsx"


# -------- Load Input Data --------
links_df = pd.read_excel(links_file, dtype=str)
product_df = pd.read_excel(product_file, dtype=str)

# Replace missing values
links_df = links_df.fillna("")
product_df = product_df.fillna("")


# -------- SQLite Connection --------
conn = sqlite3.connect(":memory:")

links_df.to_sql("links", conn, index=False, if_exists="replace")
product_df.to_sql("products", conn, index=False, if_exists="replace")

cursor = conn.cursor()


# -------- DuckDB Connection --------
duck_conn = duckdb.connect()

duck_conn.register("links", links_df)
duck_conn.register("products", products_df)


print("Initialization complete")
print("Tables loaded: links, products")

Initialization complete
Tables loaded: links, products


# Step-4 : Left join
## Datasets using : 
- Product_sample_500
- Links

In [44]:
def run_left_join(products_df, links_df, conn, output_file="product_sample_500_cleaned.xlsx"):
    """
    Perform a LEFT JOIN between the products and links tables and update missing GMID values.

    This function loads the provided DataFrames into an in-memory SQLite database,
    executes a LEFT JOIN using the product_id column, and fills missing GMID_CD
    values from the links table using SQL's COALESCE function.

    The results are written to an Excel file located in the 'output_files' folder.
    If the Excel file already exists, the corresponding sheet will be replaced,
    ensuring the output remains consistent across multiple executions.

    Parameters
    ----------
    products_df : pandas.DataFrame
        DataFrame containing product data including the GMID_CD column.

    links_df : pandas.DataFrame
        DataFrame containing reference data used to fill missing GMID_CD values.

    conn : sqlite3.Connection
        Active SQLite database connection used for executing SQL queries.

    output_file : str, optional
        Name of the Excel output file where the results will be stored.
        Default is "product_sample_500_cleaned.xlsx".

    Returns
    -------
    None
        The function writes the resulting dataset to an Excel file.
    """

    # Replace empty string values with pandas NA so they behave like NULL in SQL
    products_df = products_df.replace('', pd.NA)

    # Load the DataFrames into SQLite tables
    # Existing tables with the same name will be replaced
    products_df.to_sql("product_sample_500", conn, index=False, if_exists="replace")
    links_df.to_sql("links", conn, index=False, if_exists="replace")

    # Dictionary storing merge rules
    # Key = output sheet name, Value = SQL query to execute
    merge_rules = {
        "Left_Join": """
            SELECT 
                p.*,
                COALESCE(p.GMID_CD, l.GMID_CD) AS FILLED_GMID
            FROM product_sample_500 p
            LEFT JOIN links l
            ON p.product_id = l.product_id
        """
    }

    # Dictionary used to store results of each merge rule
    results = {}

    # Execute each SQL query defined in the merge_rules dictionary
    for rule_name, query in merge_rules.items():

        # Run the SQL query and load the result into a pandas DataFrame
        merged_df = pd.read_sql_query(query, conn)

        # Replace the original GMID_CD column with the filled values
        merged_df["GMID_CD"] = merged_df["FILLED_GMID"]

        # Remove the temporary column used during the merge process
        merged_df.drop(columns=["FILLED_GMID"], inplace=True)

        # Store the processed DataFrame using the rule name as the key
        results[rule_name] = merged_df

    # Ensure the output directory exists
    folder_path = "output_files"
    os.makedirs(folder_path, exist_ok=True)

    # Construct the full path of the output Excel file
    file_path = os.path.join(folder_path, output_file)

    # Write results to the Excel file
    # If the file exists, append mode is used and the sheet will be replaced
    # If the file does not exist, a new file will be created
    with pd.ExcelWriter(
        file_path,
        engine="openpyxl",
        mode="a" if os.path.exists(file_path) else "w",
        if_sheet_exists="replace"
    ) as writer:

        # Write each DataFrame stored in the results dictionary to Excel
        for rule_name, df in results.items():
            df.to_excel(writer, sheet_name=rule_name, index=False)


# Execute the function
run_left_join(products_df, links_df, conn)

# Step-4 : Right join
## Datasets using : 
- Product_sample_500
- Links

In [ ]:
def run_right_join(output_file="product_sample_500_cleaned.xlsx"):
    """
    Perform a RIGHT JOIN between product_sample_500 and links datasets using DuckDB.

    The function reads input Excel files, replaces empty values with NULL,
    registers the data as DuckDB tables, and performs a RIGHT JOIN based
    on the product_id column.

    Missing GMID_CD values are filled using the COALESCE SQL function.

    The result is written to an Excel file inside the 'output_files' folder.
    If the file already exists, the 'Right_Join' sheet will be replaced.

    Parameters
    ----------
    output_file : str
        Name of the Excel output file.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the RIGHT JOIN result.
    """

    # ------------------------------
    # Read input Excel files
    # ------------------------------
    product_df = pd.read_excel("Input_files/product_sample_500.csv.xlsx", dtype=str)
    links_df = pd.read_excel("Input_files/links.csv.xlsx", dtype=str)

    # Replace empty string values with NULL equivalents
    product_df = product_df.replace('', pd.NA)
    links_df = links_df.replace('', pd.NA)

    # ------------------------------
    # Create DuckDB connection
    # ------------------------------
    conn = duckdb.connect()

    # Register pandas DataFrames as DuckDB tables
    conn.register("product_sample_500", product_df)
    conn.register("links", links_df)

    # ------------------------------
    # Dictionary storing SQL logic
    # Key   → sheet name
    # Value → SQL query
    # ------------------------------
    merge_rules = {
        "Right_Join": """
        SELECT 
            l.*,
            COALESCE(l.GMID_CD, p.GMID_CD) AS FILLED_GMID
        FROM product_sample_500 p
        RIGHT JOIN links l
        ON p.product_id = l.product_id
        """
    }

    results = {}

    # Execute SQL queries from dictionary
    for rule_name, query in merge_rules.items():

        # Run SQL query
        final_links = conn.execute(query).df()

        # Replace GMID column with filled values
        final_links["GMID_CD"] = final_links["FILLED_GMID"]

        # Remove temporary column
        final_links.drop(columns=["FILLED_GMID"], inplace=True)

        results[rule_name] = final_links

    # ------------------------------
    # Output folder setup
    # ------------------------------
    output_folder = "output_files"
    os.makedirs(output_folder, exist_ok=True)

    file_path = os.path.join(output_folder, output_file)

    # ------------------------------
    # Write results to Excel
    # ------------------------------
    if os.path.exists(file_path):

        # Append mode and replace sheet if file exists
        with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            for rule_name, df in results.items():
                df.to_excel(writer, sheet_name=rule_name, index=False)

    else:

        # Create new file if it doesn't exist
        with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
            for rule_name, df in results.items():
                df.to_excel(writer, sheet_name=rule_name, index=False)

    print("Right Join result saved in:", file_path)

# Run the function
run_right_join()

Right Join result saved in: output_files\product_sample_500_cleaned.xlsx


# Step-5 : Full join
## Datasets using : 
- Product_sample_500
- Links

In [48]:
def run_full_join(output_file="product_sample_500_cleaned.xlsx"):
    """
    Perform a FULL JOIN between product_sample_500 and links datasets.

    The function reads input Excel files, replaces empty values with NULL,
    loads the datasets into an in-memory SQLite database, and performs a
    FULL JOIN using a UNION of LEFT JOIN and RIGHT JOIN.

    Missing GMID_CD values are filled using the COALESCE SQL function.
    The final dataset is written to an Excel file inside the 'output_files'
    directory.

    Parameters
    ----------
    output_file : str
        Name of the Excel file where the merged dataset will be saved.
    """

    # -----------------------------
    # File paths stored in dictionary
    # -----------------------------
    input_files = {
        "products": "Input_files/product_sample_500.csv.xlsx",
        "links": "Input_files/links.csv.xlsx"
    }

    # -----------------------------
    # Load input data
    # -----------------------------
    products_df = pd.read_excel(input_files["products"], dtype=str)
    links_df = pd.read_excel(input_files["links"], dtype=str)

    # Replace empty strings with NULL equivalents
    products_df = products_df.replace('', pd.NA)

    # -----------------------------
    # Create SQLite in-memory database
    # -----------------------------
    conn = sqlite3.connect(":memory:")

    # Load DataFrames into SQL tables
    products_df.to_sql("product_sample_500", conn, index=False, if_exists="replace")
    links_df.to_sql("links", conn, index=False, if_exists="replace")

    # -----------------------------
    # Dictionary storing SQL rules
    # Key → operation name
    # Value → SQL query
    # -----------------------------
    merge_rules = {
        "Full_Join": """
        SELECT 
            p.*,
            COALESCE(p.GMID_CD, l.GMID_CD) AS FILLED_GMID
        FROM product_sample_500 p
        LEFT JOIN links l
        ON p.product_id = l.product_id

        UNION

        SELECT 
            p.*,
            COALESCE(p.GMID_CD, l.GMID_CD) AS FILLED_GMID
        FROM product_sample_500 p
        RIGHT JOIN links l
        ON p.product_id = l.product_id
        """
    }

    # -----------------------------
    # Execute SQL rules
    # -----------------------------
    results = {}

    for rule_name, query in merge_rules.items():

        merged_df = pd.read_sql_query(query, conn)

        # Replace original GMID column
        merged_df["GMID_CD"] = merged_df["FILLED_GMID"]

        # Remove temporary column
        merged_df.drop(columns=["FILLED_GMID"], inplace=True)

        results[rule_name] = merged_df

    # -----------------------------
    # Output location
    # -----------------------------
    output_folder = "output_files"
    os.makedirs(output_folder, exist_ok=True)

    file_path = os.path.join(output_folder, output_file)

    # Write result to Excel
    for df in results.values():
        df.to_excel(file_path, index=False)


# Run function
run_full_join()